# Example 06: Error Handling & Retry（错误处理与重试）

捕获连接 / 超时 / 限流错误，指数退避重试，上下文过长时自动降级

**Demonstrates:**
- Catching common API errors (connection, timeout, rate limit, invalid request, not found)
- Exponential backoff retry for transient failures
- Graceful degradation: fall back to shorter `max_tokens` when context is too long

**Prerequisites:**
```bash
pip install openai
# Start the Hy3 server first (see quickstart.md)
```

In [ ]:
import os
import time
import logging
from openai import (
    OpenAI,
    APIConnectionError,
    APITimeoutError,
    RateLimitError,
    BadRequestError,
    NotFoundError,   # HTTP 404 — e.g. unknown model name
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)s  %(message)s")
logger = logging.getLogger(__name__)

client = OpenAI(
    base_url=os.environ.get("HY3_BASE_URL", "http://127.0.0.1:8000/v1"),
    api_key=os.environ.get("HY3_API_KEY", "EMPTY"),
    timeout=60,
)

MODEL = os.environ.get("HY3_MODEL", "hy3")

## Retry Helper（指数退避重试）

延迟序列：2s → 4s → 8s。
- **重试**：`APIConnectionError` / `APITimeoutError` / `RateLimitError`（瞬态故障）
- **不重试**：`BadRequestError` / `NotFoundError`（参数错误，重试无效）

In [ ]:
def chat_with_retry(
    messages: list,
    max_tokens: int = 512,
    max_attempts: int = 4,
    base_delay: float = 2.0,
) -> str:
    """Send a chat request with exponential backoff on transient errors."""
    for attempt in range(1, max_attempts + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0.9,
                top_p=1.0,
                max_tokens=max_tokens,
                extra_body={"chat_template_kwargs": {"reasoning_effort": "no_think"}},
            )
            return response.choices[0].message.content

        except APIConnectionError as e:
            logger.warning("Attempt %d/%d — connection error: %s", attempt, max_attempts, e)
        except APITimeoutError as e:
            logger.warning("Attempt %d/%d — timeout: %s", attempt, max_attempts, e)
        except RateLimitError as e:
            logger.warning("Attempt %d/%d — rate limited: %s", attempt, max_attempts, e)
        except (BadRequestError, NotFoundError) as e:
            # Parameter error or resource not found — fix the request, don't retry
            logger.error("Non-retryable error: %s", e)
            raise

        if attempt < max_attempts:
            delay = base_delay * (2 ** (attempt - 1))  # 2s, 4s, 8s
            logger.info("Retrying in %.0fs...", delay)
            time.sleep(delay)

    raise RuntimeError(f"All {max_attempts} attempts failed.")


def chat_with_length_fallback(messages: list, initial_max_tokens: int = 2048) -> str:
    """
    Single-fallback: halve max_tokens once and retry if context is too long.
    Raises RuntimeError if still too long after one reduction.
    """
    max_tokens = initial_max_tokens
    for attempt in range(2):  # at most 2 tries: original + one halved retry
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0.9,
                top_p=1.0,
                max_tokens=max_tokens,
                extra_body={"chat_template_kwargs": {"reasoning_effort": "no_think"}},
            )
            return response.choices[0].message.content
        except BadRequestError as e:
            err_str = str(e).lower()
            if attempt == 0 and ("context" in err_str or "token" in err_str or "length" in err_str):
                max_tokens = max_tokens // 2
                logger.warning("Context too long, retrying once with max_tokens=%d", max_tokens)
            else:
                raise
    raise RuntimeError("Could not fit request within context limits after one reduction.")

## DEMO 1: Connection Error（连接失败）

指向不存在的端口，触发 `APIConnectionError`。

In [ ]:
bad_client = OpenAI(
    base_url="http://127.0.0.1:19999/v1",
    api_key=os.environ.get("HY3_API_KEY", "EMPTY"),
    timeout=3,
)
try:
    bad_client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": "ping"}],
    )
except APIConnectionError as e:
    print(f"Caught: {type(e).__name__}")
    print("Action: check that the vLLM/SGLang server is running on the correct port.")

**Sample output:**
```
Caught: APIConnectionError
Action: check that the vLLM/SGLang server is running on the correct port.
```

## DEMO 2: Request Timeout（请求超时）

设置极短 deadline（0.001s），触发 `APITimeoutError`。

In [ ]:
tight_client = OpenAI(
    base_url=os.environ.get("HY3_BASE_URL", "http://127.0.0.1:8000/v1"),
    api_key=os.environ.get("HY3_API_KEY", "EMPTY"),
    timeout=0.001,  # impossibly short — fires before server responds
)
try:
    tight_client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": "Hello!"}],
    )
except (APITimeoutError, APIConnectionError) as e:
    print(f"Caught: {type(e).__name__}")
    print("Action: increase client timeout or use stream=True to receive tokens incrementally.")

**Sample output:**
```
Caught: APITimeoutError
Action: increase client timeout or use stream=True to receive tokens incrementally.
```

## DEMO 3: Invalid Model Name（无效模型名）

vLLM 对未知模型返回 HTTP 404，openai SDK 将其映射为 `NotFoundError`（不是 `BadRequestError`）。

In [ ]:
try:
    client.chat.completions.create(
        model="nonexistent-model-xyz",
        messages=[{"role": "user", "content": "Hello!"}],
    )
except NotFoundError as e:
    print(f"Caught: {type(e).__name__}")
    print(f"Status code: {e.status_code}")   # 404
    print(f"Message: {e.message}")
    print("Action: verify the model name matches --served-model-name.")
except APIConnectionError:
    print("Server not running — start vLLM/SGLang first to see the NotFoundError demo.")

**Sample output (server running):**
```
Caught: NotFoundError
Status code: 404
Message: The model `nonexistent-model-xyz` does not exist.
Action: verify the model name matches --served-model-name.
```

## DEMO 4: Retry Wrapper（重试封装）

正常请求通过 `chat_with_retry` 发送，验证首次成功时不会重试。

In [ ]:
messages = [{"role": "user", "content": "What is 2 + 2?"}]
try:
    answer = chat_with_retry(messages, max_tokens=32)
    print(f"Answer: {answer}")
except Exception as e:
    print(f"Failed after all retries: {e}")

**Sample output:**
```
Answer: 2 + 2 = 4.
```

## DEMO 5: Context Length Fallback（上下文过长降级）

超出上下文限制时自动将 `max_tokens` 减半并重试一次。

In [ ]:
messages = [{"role": "user", "content": "Summarize the concept of recursion in one sentence."}]
try:
    answer = chat_with_length_fallback(messages, initial_max_tokens=2048)
    print(f"Answer: {answer}")
except Exception as e:
    print(f"Failed: {e}")

**Sample output:**
```
Answer: Recursion is a technique where a function calls itself to solve
        smaller instances of the same problem until a base case is reached.
```

---

## Error Reference（错误速查）

| Exception | HTTP | 触发原因 | 是否重试 |
|---|---|---|---|
| `APIConnectionError` | — | 服务器未启动 / 端口错误 / 网络中断 | ✅ 重试 |
| `APITimeoutError` | — | 生成时间过长 / client timeout 太短 | ✅ 重试 |
| `RateLimitError` | 429 | 并发请求超过服务器限制 | ✅ 重试（退避） |
| `NotFoundError` | 404 | model 名错误（不匹配 `--served-model-name`） | ❌ 不重试（修 model 名） |
| `BadRequestError` | 400 | 参数格式错误 / 上下文过长 | ❌ 不重试（修参数） |